**420-A58-SF - Algorithmes d'apprentissage non supervisé - Hiver 2026 - Spécialisation technique en Intelligence Artificielle**<br/>
MIT License - Copyright (c) 2026 Mikaël Swawola

# Évaluation formative #2 (solutions)

| | |
|---|---|
| **Durée** | 1 heure |
| **Travail** | Individuel |
| **Documents** | Autorisés |
| **Remise** | Remettre ce notebook complété sur Lea |

## Barème

| Section | Question | Points |
|---------|----------|--------|
| 1 - Produits scalaires et encodage binaire | Q1.1 - Produit scalaire matrice-vecteur | 1 |
| | Q1.2 - Encodage par le signe | 1 |
| | Q1.3 - Conversion binaire → décimal | 1 |
| 2 - Distance cosinus et plus proches voisins | Q2.1 - Calcul de la distance cosinus | 2 |
| | Q2.2 - Recherche du plus proche voisin | 2 |
| 3 - Tables de hachage et bins | Q3.1 - Regroupement de documents dans des bins | 2 |
| | Q3.2 - Recherche dans un bin et limites | 1 |
| **Total** | | **10** |

In [1]:
import numpy as np

---
## 1 - Produits scalaires et encodage binaire (3 points)

Dans l'algorithme LSH par projection binaire aléatoire, chaque document est projeté sur un ensemble de vecteurs aléatoires. Le **signe** de chaque produit scalaire détermine un **bit** (0 ou 1). L'ensemble de ces bits forme un **index de bin**.

Soit la matrice de 3 vecteurs aléatoires (en lignes) et un vecteur document :

In [2]:
# Vecteurs aléatoires (3 vecteurs de dimension 4)
R = np.array([[ 0.5, -1.2,  0.3,  0.8],
              [-0.7,  0.4,  1.1, -0.5],
              [ 0.2,  0.9, -0.6,  0.1]])

# Vecteur document (dimension 4)
d = np.array([1.0, 2.0, 0.5, -1.0])

**Q1.1 - Calculer le produit scalaire entre chaque vecteur aléatoire et le vecteur document `d`. Stocker les résultats dans un tableau NumPy nommé `projections`.** (1 point)

*Indice : le résultat doit être un tableau de 3 valeurs.*

In [3]:
projections = R @ d
print(projections)  # [-2.55  1.15  1.6 ]

[-2.55  1.15  1.6 ]


**Q1.2 - À partir du tableau `projections`, créer un tableau `bits` contenant 1 si la projection est positive ou nulle, et 0 sinon.** (1 point)

*Indice : utiliser une comparaison vectorisée.*

In [4]:
bits = (projections >= 0).astype(int)
print(bits)  # [0 1 1]

[0 1 1]


**Q1.3 - Convertir le tableau `bits` en un entier décimal représentant l'index du bin. Par exemple, `[1, 0, 1]` correspond à $1 \times 2^2 + 0 \times 2^1 + 1 \times 2^0 = 5$. Stocker le résultat dans une variable `bin_index`.** (1 point)

*Indice : vous pouvez utiliser `np.dot` avec un vecteur de puissances de 2.*

In [5]:
powers_of_two = 1 << np.arange(len(bits) - 1, -1, -1)  # [4, 2, 1]
bin_index = bits.dot(powers_of_two)
print(bin_index)  # 3  (0*4 + 1*2 + 1*1 = 3)

3


---
## 2 - Distance cosinus et plus proches voisins (4 points)

La distance cosinus est la métrique utilisée pour comparer les documents dans l'espace TF-IDF. Elle est définie par :

$$d_{\cos}(\mathbf{a}, \mathbf{b}) = 1 - \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \, \|\mathbf{b}\|}$$

Voici un petit corpus de 5 documents représentés par des vecteurs de dimension 3 :

In [6]:
corpus = np.array([
    [1.0, 0.0, 0.0],   # doc 0
    [1.0, 1.0, 0.0],   # doc 1
    [0.0, 1.0, 1.0],   # doc 2
    [0.5, 0.5, 0.0],   # doc 3
    [0.0, 0.0, 1.0],   # doc 4
])

requete = np.array([1.0, 0.8, 0.0])  # document requête

**Q2.1 - Écrire une fonction `distance_cosinus(a, b)` qui calcule la distance cosinus entre deux vecteurs NumPy `a` et `b`. Tester votre fonction en calculant la distance entre `requete` et le document 0.** (2 points)

In [7]:
def distance_cosinus(a, b):
    return 1 - np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(distance_cosinus(requete, corpus[0]))  # 0.2191

0.21913119055696972


**Q2.2 - En utilisant votre fonction `distance_cosinus`, calculer la distance entre `requete` et chaque document du corpus. Identifier l'indice du document le plus proche (distance minimale). Afficher l'indice et la distance.** (2 points)

In [8]:
distances = np.array([distance_cosinus(requete, corpus[i]) for i in range(len(corpus))])
idx_min = np.argmin(distances)
print(f"Document le plus proche : {idx_min}, distance : {distances[idx_min]:.4f}")
# Document le plus proche : 1, distance : 0.0061
# Note: doc 1 et doc 3 sont a la meme distance (0.0061) car proportionnels

Document le plus proche : 1, distance : 0.0061


---
## 3 - Tables de hachage et bins (3 points)

Dans LSH, les documents sont regroupés dans des **bins** à l'aide d'une table de hachage (dictionnaire Python). Chaque bin contient la liste des indices des documents qui partagent le même index.

Voici les index de bins (décimaux) attribués à chaque document du corpus :

In [9]:
# Index de bin pour chaque document (déjà calculés)
bin_indices = [3, 2, 5, 3, 7]  # doc 0 -> bin 3, doc 1 -> bin 2, doc 2 -> bin 5, doc 3 -> bin 3, doc 4 -> bin 7

**Q3.1 - Construire un dictionnaire `table` où chaque clé est un index de bin et chaque valeur est la liste des indices des documents appartenant à ce bin.** (2 points)

Résultat attendu : `{3: [0, 3], 2: [1], 5: [2], 7: [4]}`

In [10]:
table = {}
for doc_id, bin_idx in enumerate(bin_indices):
    if bin_idx not in table:
        table[bin_idx] = []
    table[bin_idx].append(doc_id)

print(table)  # {3: [0, 3], 2: [1], 5: [2], 7: [4]}

{3: [0, 3], 2: [1], 5: [2], 7: [4]}


**Q3.2 - Supposons que le document requête tombe dans le bin 3. Quels documents sont retournés comme candidats ? Parmi eux, lequel est réellement le plus proche de `requete` selon la distance cosinus ? Expliquer en une phrase pourquoi LSH pourrait manquer le vrai plus proche voisin.** (1 point)

In [11]:
candidats = table[3]  # [0, 3]
for c in candidats:
    print(f"Doc {c} : distance = {distance_cosinus(requete, corpus[c]):.4f}")
# Doc 0 : distance = 0.2191
# Doc 3 : distance = 0.0061

Doc 0 : distance = 0.2191
Doc 3 : distance = 0.0061


**Réponse Q3.2:** Les candidats du bin 3 sont les documents 0 et 3. Le document 3 est le plus proche parmi les candidats (distance = 0.006). Cependant, le document 1 (dans le bin 2) est tout aussi proche (distance = 0.006) mais se retrouve dans un bin différent. LSH pourrait manquer le vrai plus proche voisin car la projection aléatoire peut séparer des documents similaires dans des bins différents — c'est une méthode **approximative**.

---
**Fin de l'évaluation formative #2**